# 5장. 후보 모델 배포 확인과 되돌림 조건

이 notebook은 1장부터 4장까지의 증거를 `risk-classifier@candidate`의 배포/보류/되돌림 확인 결과로 연결하는 최종 실습입니다. 입력 분포, score와 prediction 분포, 운영 오류, Argo CD GitOps 확인 결과, KServe readiness, 최종 체크리스트를 같은 배포 확인으로 묶습니다.


## 1. 실행 환경과 확인 결과 path 고정

### 1-1. 최종 확인에 사용할 artifact 범위 확인

이 셀에서는 최종 배포 의견이 어떤 확인 결과 file에 근거하는지 고정하는 것입니다. 5장은 새 도구를 배우는 장이 아니라, 앞 장에서 만든 산출물을 감사 추적 가능한 배포/보류/되돌림 확인으로 묶는 장입니다.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
lite = prepared.aiq_lite

for name in runtime.LITE_NAMES:
    globals()[name] = getattr(lite, name)

artifact_contract = [
    ("data_quality", "chapter_01_quality_report.md", "평가 가능성 확인이며 운영 current 정상으로 확대하지 않습니다."),
    ("model_quality", "model_test_eval.json", "선택된 모델과 threshold의 test 평가로 한정합니다."),
    ("validation_comparison", "validation_degradation_comparison.json", "validation 품질 저하 비교이며 운영 root cause 확정이 아닙니다."),
    ("operational_current", "drift_report.md, quality_issue_trace.md", "current batch 입력 구성 변화와 운영 검증 실패 후보입니다."),
    ("gitops_serving", "Argo CD Application, KServe InferenceService", "배포 의도와 live sync 확인 결과를 분리합니다."),
    ("deployment_check", "release_approval.md, ai_qa_checklist.md", "실패 기준과 미검증 항목을 확인 결과 path로 남깁니다."),
]

print("[notebook role]")
print(f"- package_module: {lite.__name__}")
print("- notebook_role: final_deployment_evidence_workbook")
print("- release_context: 배포 회의 전 확인 결과 정리")
print()
print("[읽을 산출물과 근거 경계]")
for stage, artifact, boundary in artifact_contract:
    print(f"- {stage}: {artifact} | {boundary}")


[notebook role]
- package_module: ai_quality.lite
- notebook_role: final_deployment_evidence_workbook
- release_context: 배포 회의 전 확인 결과 정리

[읽을 산출물과 근거 경계]
- data_quality: chapter_01_quality_report.md | 평가 가능성 확인이며 운영 current 정상으로 확대하지 않습니다.
- model_quality: model_test_eval.json | 선택된 모델과 threshold의 test 평가로 한정합니다.
- validation_comparison: validation_degradation_comparison.json | validation 품질 저하 비교이며 운영 root cause 확정이 아닙니다.
- operational_current: drift_report.md, quality_issue_trace.md | current batch 입력 구성 변화와 운영 검증 실패 후보입니다.
- gitops_serving: Argo CD Application, KServe InferenceService | 배포 의도와 live sync 확인 결과를 분리합니다.
- deployment_check: release_approval.md, ai_qa_checklist.md | 실패 기준과 미검증 항목을 확인 결과 path로 남깁니다.


이 출력에서 확인할 핵심은 최종 확인의 출처를 섞지 않는 것입니다. 운영 current에서 보인 `high_risk` 증가를 test 성능 저하로 쓰면 안 되고, validation 비교를 운영 원인 확정으로 쓰면 안 됩니다.

## 2. 입력 데이터 분포 변화 확인

### 2-1. 기준 요청과 current 요청 data brief

이 셀에서는 current batch 입력이 operational baseline과 어떻게 다른지 확인하는 것입니다. 한 row는 serving request sample 하나이며, 이 데이터는 정답 label 평가가 아니라 운영 입력 feature 변화 확인에 사용합니다.

입력 분포 변화는 모델 문제의 확정 증거가 아닙니다. 하지만 score와 prediction 변화가 함께 나타나면 원인 후보로 강해집니다.

먼저 source, row grain, row count, feature count를 고정한 뒤 feature 평균 변화를 확인합니다.


In [2]:
fallback_requests = sample_vital_signs().head(120)
baseline_requests, baseline_source = load_csv_or_sample(
    "data/serving_requests_valid.csv",
    fallback_requests,
    nrows=None,
)
current_requests, current_source = load_csv_or_sample(
    "data/serving_requests_current.csv",
    fallback_requests,
    nrows=None,
)

print("[입력 데이터 범위]")
print(f"- serving_requests_valid: {baseline_source}, rows={len(baseline_requests)}, grain=one serving request sample")
print(f"- serving_requests_current: {current_source}, rows={len(current_requests)}, grain=one serving request sample")
print(f"- feature_count: {len(FEATURE_COLUMNS)}")

feature_comparison = compare_input_distribution(baseline_requests, current_requests, FEATURE_COLUMNS)
shifted_features = feature_comparison.loc[feature_comparison["shifted"], "feature"].tolist()

display(feature_comparison.sort_values(["shifted", "feature"], ascending=[False, True]))
print(f"shifted_features={shifted_features or 'none'}")


[입력 데이터 범위]
- serving_requests_valid: embedded JupyterLite sample, rows=12, grain=one serving request sample
- serving_requests_current: embedded JupyterLite sample, rows=12, grain=one serving request sample
- feature_count: 6


,feature,baseline_mean,current_mean,mean_delta,delta_ratio,shifted
2,body_temperature,37.3083,37.3083,0.0,0.0,False
5,diastolic_blood_pressure,84.6667,84.6667,0.0,0.0,False
0,heart_rate,98.9091,98.9091,0.0,0.0,False
3,oxygen_saturation,94.1667,94.1667,0.0,0.0,False
1,respiratory_rate,21.5000,21.5000,0.0,0.0,False
4,systolic_blood_pressure,135.5000,135.5000,0.0,0.0,False


shifted_features=none


이 출력에서 확인할 핵심은 shifted feature가 있는지와 그 방향입니다. 입력 분포 변화가 보이면 “모델이 바뀌었다”가 아니라 “입력 조건이 baseline과 달라졌으므로 동일 기준 평가가 필요한 상태”라고 표현합니다.

feature 변화가 있어도 바로 drift 확정이라고 쓰지 않습니다. current batch 기간, 상위 데이터 수집 경로, 배포 변경 여부를 함께 확인해야 합니다.

### 2-2. drift report artifact와 notebook 계산 비교

이 셀에서는 notebook에서 계산한 feature 변화가 prepared `drift_report.md`와 같은 해석 흐름에 있는지 확인하는 것입니다. 최종 보고서는 즉석 계산보다 제출용 artifact를 근거로 삼아야 합니다.

artifact를 읽는 목적은 문장을 복사하는 것이 아니라, 어떤 확인 결과 path가 최종 체크리스트와 연결되는지 확인하는 것입니다.

이 출력에서는 drift report가 존재하고, notebook의 shifted feature 후보와 같은 종류의 확인 결과를 담고 있는지 확인하는 것입니다.


In [3]:
drift_report_text = runtime.read_text_artifact("artifacts/reports/drift_report.md")
drift_report_checks = {
    "drift_report_exists": bool(drift_report_text.strip()),
    "shifted_features_from_notebook": ", ".join(shifted_features) or "none",
    "mentions_heart_rate": "heart_rate" in drift_report_text,
    "mentions_oxygen_saturation": "oxygen_saturation" in drift_report_text,
}

print("[drift_report.md preview]")
print("\n".join(drift_report_text.splitlines()[:12]))
print("\n[report checks]")
for check, observed in drift_report_checks.items():
    print(f"- {check}: {observed}")


[drift_report.md preview]
# Drift Report

## Input Distribution

| feature | baseline_mean | current_mean | delta | delta_ratio | shifted |
| --- | ---: | ---: | ---: | ---: | --- |
| heart_rate | 79.3417 | 89.8333 | 10.4917 | 0.1322 | True |
| respiratory_rate | 15.8417 | 15.7083 | -0.1333 | -0.0084 | False |
| body_temperature | 36.7583 | 36.7128 | -0.0455 | -0.0012 | False |
| oxygen_saturation | 97.5631 | 96.0934 | -1.4698 | -0.0151 | True |
| systolic_blood_pressure | 123.9000 | 123.5583 | -0.3417 | -0.0028 | False |
| diastolic_blood_pressure | 80.3333 | 79.5583 | -0.7750 | -0.0096 | False |

[report checks]
- drift_report_exists: True
- shifted_features_from_notebook: none
- mentions_heart_rate: True
- mentions_oxygen_saturation: True


이 출력에서 확인할 핵심은 notebook 계산과 제출 artifact가 같은 확인을 가리키는지입니다. 직접 계산하지 않은 경우에도 prepared artifact에서 확인한 값이라고 보고서에 명시하면 됩니다.

## 3. score와 prediction 분포 비교

### 3-1. 운영 로그에서 score와 `high_risk` 비율 변화 확인

이 셀에서는 입력 변화가 output 변화와 함께 나타나는지 확인하는 것입니다. score는 prediction 이전의 연속값이고, prediction은 threshold를 적용한 최종 class입니다.

운영 로그는 정답 label이 없는 current 관측일 수 있습니다. 따라서 Precision/Recall을 바로 말하지 않고, score 평균, `high_risk` rate, validation failure, latency 같은 proxy signal을 봅니다.

이 단계의 출력은 입력 분포 변화와 연결해 읽습니다. feature 변화, score 변화, prediction 변화가 같은 방향이면 데이터 조건 변화와 모델 출력 변화가 연결된 후보로 기록합니다.


In [4]:
def read_jsonl_artifact(path: str, max_events: int | None = 120) -> list[dict[str, object]]:
    text = runtime.read_text_artifact(path)
    events = [json.loads(line) for line in text.splitlines() if line.strip()]
    return events[:max_events] if max_events is not None else events


baseline_events = read_jsonl_artifact("artifacts/logs/chapter_04_normal_events.jsonl")
current_events = read_jsonl_artifact("artifacts/logs/chapter_04_anomaly_events.jsonl")
baseline_snapshot = quality_snapshot(baseline_events)
current_snapshot = quality_snapshot(current_events)
score_comparison = score_distribution_comparison(baseline_events, current_events)
score_comparison_table = pd.DataFrame([score_comparison])
snapshot_table = pd.DataFrame(
    [
        {"signal": key, "baseline": baseline_snapshot.get(key), "current": current_snapshot.get(key)}
        for key in sorted(set(baseline_snapshot) | set(current_snapshot))
    ]
)

display(snapshot_table)
display(score_comparison_table)


,signal,baseline,current
0,average_latency_ms,103.7500,223.7500
1,average_score,0.5016,0.6411
2,error_count,0.0000,8.0000
3,error_rate,0.0000,0.0667
4,high_risk_count,26.0000,55.0000
5,high_risk_rate,0.2167,0.4583
6,low_risk_count,94.0000,65.0000
7,request_count,120.0000,120.0000
8,valid_high_risk_rate,0.2167,0.4643
9,valid_request_count,120.0000,112.0000


,baseline_average_score,current_average_score,average_score_delta,baseline_high_risk_rate,current_high_risk_rate,high_risk_rate_delta
0,0.5016,0.6411,0.1395,0.2167,0.4583,0.2416


이 출력에서 확인할 핵심은 `high_risk` 비율 상승만 단독으로 쓰지 않는 것입니다. 평균 score 변화, threshold 유지 여부, validation failure, latency를 함께 봐야 운영자가 같은 결론을 재현할 수 있습니다.

### 3-2. 원인 후보를 owner와 next action으로 정리

이 셀에서는 입력 변화, prediction shift, API validation, service latency를 각각 다른 owner와 next action으로 분리하는 것입니다. 후보를 하나로 합치면 후속 조치가 흐려집니다.

`trace_candidates`는 앞 셀의 feature comparison, score comparison, operational delta를 받아 후보 표를 만듭니다. 이 helper는 결론을 자동으로 승인하지 않고, 보고서에 필요한 owner와 next action을 정리하는 역할입니다.

이 표는 `quality_issue_trace.md`와 함께 최종 체크리스트에 들어갈 확인 결과입니다.


In [5]:
quality_report = compare_snapshots(baseline_snapshot, current_snapshot)
candidates = trace_candidates(feature_comparison, score_comparison, quality_report, current_events)
quality_issue_trace_text = runtime.read_text_artifact("artifacts/reports/quality_issue_trace.md")
quality_delta_table = pd.DataFrame(
    [
        {"signal": "error_rate_delta", "observed": quality_report["error_rate_delta"], "candidate": "api_validation"},
        {"signal": "latency_delta_ms", "observed": quality_report["latency_delta_ms"], "candidate": "service_latency"},
        {"signal": "high_risk_rate_delta", "observed": quality_report["high_risk_rate_delta"], "candidate": "prediction_shift"},
        {"signal": "average_score_delta", "observed": quality_report["average_score_delta"], "candidate": "score_distribution_shift"},
    ]
)

display(quality_delta_table)
display(candidates)
print("\\n".join(quality_issue_trace_text.splitlines()[:10]))


,signal,observed,candidate
0,error_rate_delta,0.0667,api_validation
1,latency_delta_ms,120.0000,service_latency
2,high_risk_rate_delta,0.2416,prediction_shift
3,average_score_delta,0.1395,score_distribution_shift


,candidate,evidence,owner,next_action
0,prediction_shift,high_risk_rate_delta=0.2416,ML Engineering,점수 분포와 임계값 설정을 비교합니다.
1,api_validation,request_id=current-0000; failed_field=oxygen_s...,Client Integration,검증 실패 필드와 입력 출처를 확인합니다.
2,service_latency,latency_delta_ms=120.0,Platform/MLOps,서비스 부하와 배포 상태를 확인합니다.


# Quality Issue Trace\n\n| category | evidence | owner | audit_reference | next_action |\n| --- | --- | --- | --- | --- |\n| input_case_mix_shift | shifted_features=heart_rate, oxygen_saturation | Data Engineering | artifacts/reports/drift_report.md#input-distribution | 최근 입력 출처와 전처리 변경을 확인합니다. |\n| prediction_shift | high_risk_rate_delta=0.2417 | ML Engineering | artifacts/reports/drift_report.md#score-and-prediction-distribution | 점수 분포와 임계값 설정을 비교합니다. |\n| api_validation | error_rate_delta=0.0667 | Client Integration | request_id=current-0000, client_id=partner-feed-v2, source_system=upstream-partner-feed, failed_field=oxygen_saturation | 검증 실패 예시에서 failed_field, client_id, source_system을 확인하고 Client Integration owner에게 전달합니다. |\n| service_latency | latency_delta_ms=120.0 | Platform/MLOps | artifacts/grafana/ai_quality_overview_dashboard.json#average-latency | 서비스 부하, 의존성 지연, Pod 상태를 확인합니다. |


이 출력에서 확인할 핵심은 원인 후보가 조치 가능한 단위로 바뀌었다는 점입니다. input case mix는 Data Engineering, API validation은 Client Integration, latency는 Platform/MLOps처럼 owner가 나뉩니다.

## 4. 배포 확인과 승인 조건 확인

### 4-1. 배포 확인 예제 데이터와 label basis 확인

이 셀에서는 배포 확인에 들어가는 데이터의 label basis를 먼저 확인하는 것입니다. 모델 품질 지표를 계산하거나 배포 확인 리포트를 읽기 전에 label 허용값과 class support를 확인해야 합니다.

`release_regression_cases.csv`는 최종 승인 확인 예제에 쓰는 작은 regression case 묶음입니다. 한 row는 회귀 테스트 sample이고, `label`은 평가 기준 정답입니다.

이 단계에서는 모델 개선을 목표로 하지 않습니다. label basis가 지표 해석에 충분한지, 그리고 제출용 `label_basis_check.md`가 같은 근거를 남기는지 확인합니다.


In [6]:
release_cases, deployment_case_source = load_csv_or_sample(
    "data/release_regression_cases.csv",
    sample_vital_signs(),
    nrows=None,
)
release_label_distribution = (
    release_cases["label"]
    .value_counts(dropna=False)
    .rename("count")
    .reset_index()
    .rename(columns={"index": "label"})
    .assign(ratio_pct=lambda table: (table["count"] / len(release_cases) * 100).round(2))
)
label_basis_text = runtime.read_text_artifact("artifacts/reports/label_basis_check.md")

print("[배포 확인 예제 데이터]")
print(f"- source: {deployment_case_source}")
print("- row_grain: one release check sample")
print(f"- row_count: {len(release_cases)}")
print(f"- label_values: {', '.join(sorted(release_cases['label'].dropna().astype(str).unique()))}")

display(release_label_distribution)
print("\n[label_basis_check.md preview]")
print("\n".join(label_basis_text.splitlines()[:12]))


[배포 확인 예제 데이터]
- source: embedded JupyterLite sample
- row_grain: one release check sample
- row_count: 12
- label_values: high_risk, low_risk


,label,count,ratio_pct
0,low_risk,6,50.0
1,high_risk,6,50.0



[label_basis_check.md preview]
# Label Basis Check

| source | target_column | allowed_labels | label_mapping | evaluation_ready |
| --- | --- | --- | --- | --- |
| data/release_regression_cases.csv | label | high_risk, low_risk | High Risk->high_risk, Low Risk->low_risk, high_risk->high_risk, low_risk->low_risk | True |

| label | count |
| --- | ---: |
| high_risk | 37 |
| low_risk | 33 |

| positive_label | positive_count | negative_label | negative_count | invalid_count | missing_count | positive_rate |


이 출력에서 확인할 핵심은 배포 확인의 지표가 어떤 label support에서 나온 것인지 설명할 수 있어야 한다는 점입니다. label basis가 불명확하면 `inconclusive` 또는 보류 확인을 남기는 것이 안전합니다.

### 4-2. 배포 기준 결과와 unresolved risk 확인

이 셀에서는 승인 여부보다 실패 기준과 미검증 리스크를 먼저 읽는 것입니다. `approved=False` 자체보다 어떤 기준이 실패했고 어떤 확인 결과가 없는지가 중요합니다.

이 notebook은 최종 확인의 기준 산출물을 `release_approval.md`로 둡니다. prepared API contract가 통과했더라도 live 배포 확인 결과가 없으면 운영 배포 검증 완료라고 말하지 않습니다.

이 표에서는 제출용 배포 확인 리포트가 어떤 check를 통과하거나 실패했는지 확인하는 것입니다. Notebook 계산과 보고서 artifact가 다른 값을 만들면 audit에서 방어되지 않으므로, 이 셀은 report artifact를 직접 파싱합니다.


In [7]:
release_approval_text = runtime.read_text_artifact("artifacts/reports/release_approval.md")


def parse_markdown_table_rows(text: str, header_start: str) -> list[dict[str, str]]:
    lines = text.splitlines()
    start = next(index for index, line in enumerate(lines) if line.startswith(header_start))
    rows: list[list[str]] = []
    for line in lines[start:]:
        if not line.startswith("|"):
            if rows:
                break
            continue
        rows.append([cell.strip() for cell in line.strip("|").split("|")])
    header = rows[0]
    return [dict(zip(header, row)) for row in rows[2:] if len(row) == len(header)]


release_summary_row = parse_markdown_table_rows(
    release_approval_text,
    "| recommendation | approved | failed_checks | unresolved_risks |",
)[0]
release_checks = parse_markdown_table_rows(
    release_approval_text,
    "| check | observed | criterion | result |",
)
unresolved_risks = parse_markdown_table_rows(
    release_approval_text,
    "| area | status | 확인 결과 | owner | next_action |",
)
release_report_summary = {
    "recommendation": release_summary_row["recommendation"],
    "approved": release_summary_row["approved"] == "True",
    "failed_checks": [
        item.strip()
        for item in release_summary_row["failed_checks"].split(",")
        if item.strip() and item.strip() != "-"
    ],
    "unresolved_risks": [
        item.strip()
        for item in release_summary_row["unresolved_risks"].split(",")
        if item.strip() and item.strip() != "-"
    ],
}
precision_result = next(row["result"] for row in release_checks if row["check"] == "precision")

print("[배포 확인 요약]")
for key in ["failed_checks", "unresolved_risks"]:
    print(f"- {key}: {release_summary_row[key]}")
print()
print("[기준별 결과]")
for row in release_checks:
    print(f"- {row['check']}: {row['result']} | observed={row['observed']} | criterion={row['criterion']}")
print()
print("[미확인 항목]")
for row in unresolved_risks:
    print(f"- {row['area']}: {row['status']} | owner={row['owner']}")
print()
print("[report consistency]")
print(f"- precision_pass_in_report: {precision_result}")
print(f"- failed_checks_from_report: {', '.join(release_report_summary['failed_checks'])}")
print(f"- unresolved_risks_from_report: {', '.join(release_report_summary['unresolved_risks'])}")
print("\n".join(release_approval_text.splitlines()[:18]))


[배포 확인 요약]
- failed_checks: recall, error_rate
- unresolved_risks: live_deployment

[기준별 결과]
- precision: pass | observed=1.0000 | criterion=>= 0.6000
- recall: fail | observed=0.5926 | criterion=>= 0.6000
- error_rate: fail | observed=0.0667 | criterion=<= 0.0500
- latency: pass | observed=223.7500 | criterion=<= 250.0000 ms
- prepared_api_contract: pass | observed=True | criterion=local/prepared contract check is True

[미확인 항목]
- live_deployment: unverified | owner=Platform/MLOps

[report consistency]
- precision_pass_in_report: pass
- failed_checks_from_report: recall, error_rate
- unresolved_risks_from_report: live_deployment
# Deployment Decision Report

## Decision Summary

| recommendation | approved | failed_checks | unresolved_risks | re_evaluation_condition |
| --- | --- | --- | --- | --- |
| conditional_hold | False | recall, error_rate | live_deployment | failed_checks와 unresolved_risks가 해소되고 owner별 확인 결과가 같은 배포 기준을 만족하면 재평가합니다. |

| approved | failed_checks | notes |
| ---

### 4-3. Argo CD와 KServe 확인 결과 gap 확인

이 셀에서는 배포 확인에 GitOps 배포 확인 결과가 있는지 확인하는 것입니다. prepared report의 metric 기준이 실패하지 않더라도, Argo CD sync revision과 KServe readiness가 없으면 운영 전환 완료로 쓸 수 없습니다.


In [8]:
def find_optional_file(candidates: list[str]) -> tuple[Path | None, str]:
    roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in roots:
        for candidate in candidates:
            path = (root / candidate).resolve()
            if path.exists():
                return path, path.read_text(encoding="utf-8")
    return None, ""

argocd_path, argocd_text = find_optional_file(
    [
        "demos/ch03_docker_kubernetes/argocd/application.yaml",
        "artifacts/deployment/chapter_03/argocd/application.yaml",
        "../artifacts/deployment/chapter_03/argocd/application.yaml",
    ]
)
kserve_path, kserve_text = find_optional_file(
    [
        "demos/ch03_docker_kubernetes/gitops/base/inferenceservice.yaml",
        "artifacts/deployment/chapter_03/gitops/base/inferenceservice.yaml",
        "../artifacts/deployment/chapter_03/gitops/base/inferenceservice.yaml",
    ]
)

gitops_checks = [
    ("Argo CD Application manifest", "checked" if argocd_path else "missing", str(argocd_path) if argocd_path else "not found"),
    ("Argo CD source path", "checked" if "gitops/overlays/student" in argocd_text else "missing", "spec.source.path"),
    ("Argo CD live sync", "unverified", "requires argocd app sync output"),
    ("KServe InferenceService manifest", "checked" if kserve_path else "missing", str(kserve_path) if kserve_path else "not found"),
    ("KServe candidate alias", "checked" if "ai-quality/model-alias: candidate" in kserve_text else "missing", "metadata label"),
    ("KServe readiness", "unverified", "requires kubectl get inferenceservice"),
]

print("[GitOps/KServe 확인]")
for check, status, result in gitops_checks:
    print(f"- {check}: {status} | {result}")


[GitOps/KServe 확인]
- Argo CD Application manifest: checked | /Users/seungbaeji/Workspace/tta/tta-aiqa/demos/ch03_docker_kubernetes/argocd/application.yaml
- Argo CD source path: checked | spec.source.path
- Argo CD live sync: unverified | requires argocd app sync output
- KServe InferenceService manifest: checked | /Users/seungbaeji/Workspace/tta/tta-aiqa/demos/ch03_docker_kubernetes/gitops/base/inferenceservice.yaml
- KServe candidate alias: checked | metadata label
- KServe readiness: unverified | requires kubectl get inferenceservice


이 출력에서 확인할 핵심은 manifest 확인 결과와 live 확인 결과를 분리하는 것입니다. `Application`과 `InferenceService` 파일은 checked여도, `argocd app sync`와 `kubectl get inferenceservice`를 실행하지 않았다면 live 배포 확인 결과는 `unverified`로 남깁니다.


이 출력에서 확인할 핵심은 조건부 보류가 실패가 아니라 방어 가능한 품질 해석이라는 점입니다. Recall 또는 error rate 기준이 실패하고 live 확인 결과가 없으면 승인 대신 재평가 조건을 남깁니다.

prepared report가 있으므로 최종 보고서에는 “notebook helper로 확인한 rule 흐름”과 “prepared artifact에서 확인한 제출 산출물”을 구분해서 적습니다.

## 5. 최종 AI QA 체크리스트 작성

### 5-1. 체크리스트 상태와 확인 결과 path 확인

이 셀에서는 체크리스트가 단순 완료 목록이 아니라 배포 확인 문서인지 확인하는 것입니다. 각 항목에는 상태, 확인 결과, QA 코멘트, 담당, 다음 조치가 있어야 합니다.

`ai_qa_checklist.md`는 이번 사건 제출용 점검본이고, `ai_qa_checklist_template.md`는 반복 사용 템플릿입니다. 제출본은 `pass`, `fail`, `unverified`, `보류`를 구분해야 합니다.

이 표에서는 notebook에서 만든 후보와 prepared checklist의 핵심 상태가 연결되는지 확인하는 것입니다.


In [9]:
checklist_text = runtime.read_text_artifact("artifacts/reports/ai_qa_checklist.md")
checklist = [
    ("input distribution", "blocked" if bool(feature_comparison["shifted"].any()) else "ready", "Data Engineering", "drift_report.md"),
    ("score and prediction distribution", "blocked" if score_comparison["high_risk_rate_delta"] > 0.15 else "ready", "ML Engineering", "drift_report.md"),
    ("api validation failures", "blocked" if quality_report["error_rate_delta"] > 0.03 else "ready", "Client Integration", "quality_issue_trace.md"),
    ("service latency", "blocked" if quality_report["latency_delta_ms"] > 100 else "ready", "Platform/MLOps", "quality_issue_trace.md"),
    ("live deployment evidence", "blocked", "Platform/MLOps", "release_approval.md"),
    ("release report summary", "blocked" if not release_report_summary["approved"] else "ready", "QA Lead", "release_approval.md"),
]
checklist_artifact_check = {
    "deployment_ready_state": "배포 준비 상태: blocked" in checklist_text,
    "conditional_hold_status_present": "conditional_hold" in checklist_text,
    "live_deployment_unverified": "live_deployment" in checklist_text and "unverified" in checklist_text,
    "owner_next_action": "다음 조치" in checklist_text,
}
deployment_ready = not any(status == "blocked" for _, status, _, _ in checklist)

print("[최종 체크리스트]")
for item, status, owner, artifact in checklist:
    print(f"- {item}: {status} | owner={owner} | 확인 결과={artifact}")
print()
print("[체크리스트 산출물 확인]")
for check, observed in checklist_artifact_check.items():
    print(f"- {check}: {observed}")
print("deployment_ready", deployment_ready)


[최종 체크리스트]
- input distribution: ready | owner=Data Engineering | 확인 결과=drift_report.md
- score and prediction distribution: blocked | owner=ML Engineering | 확인 결과=drift_report.md
- api validation failures: blocked | owner=Client Integration | 확인 결과=quality_issue_trace.md
- service latency: blocked | owner=Platform/MLOps | 확인 결과=quality_issue_trace.md
- live deployment evidence: blocked | owner=Platform/MLOps | 확인 결과=release_approval.md
- release report summary: blocked | owner=QA Lead | 확인 결과=release_approval.md

[체크리스트 산출물 확인]
- deployment_ready_state: True
- conditional_hold_status_present: True
- live_deployment_unverified: True
- owner_next_action: True
deployment_ready False


이 출력에서 확인할 핵심은 checklist가 `blocked` 상태와 미검증 항목을 숨기지 않는다는 점입니다. 누락된 live 배포 확인 결과를 숨기면 배포 이후 문제가 생겼을 때 QA 해석을 방어하기 어렵습니다.

### 5-2. 최종 확인 항목과 재평가 조건 정리

마지막 정리는 보고 문장을 대신 만들어 주는 단계가 아니라, 앞에서 확인한 항목을 재평가 조건으로 묶는 단계입니다. Notebook은 결론 문장을 출력하지 않고, 입력 변화, 예측 변화, API 검증 실패, live 배포 확인 결과의 남은 항목만 보여 줍니다.

수강생은 이 출력과 prepared report artifact를 읽고 자신의 문장으로 정리합니다. 코드가 “승인합니다” 또는 “보류합니다”를 대신 말해 주면 실무 보고서 작성 연습이 약해지므로, 이 셀은 확인 항목과 owner, next action만 남깁니다.


In [10]:
owner_actions = candidates.loc[:, ["candidate", "owner", "next_action"]].to_dict("records")
final_check_result_packet = [
    ("input_shift", ", ".join(shifted_features) or "none", "Data Engineering", "데이터 수집 경로와 upstream feed 변경 여부 확인"),
    ("prediction_shift", f"high_risk_rate_delta={score_comparison['high_risk_rate_delta']:.4f}", "ML Engineering", "threshold와 score distribution 재확인"),
    ("api_validation", f"error_rate_delta={quality_report['error_rate_delta']}", "Client Integration", "failed_field와 client payload 변경 확인"),
    (
        "deployment_check",
        f"failed_checks={release_report_summary['failed_checks']}, unresolved_risks={release_report_summary['unresolved_risks']}",
        "QA Lead",
        "같은 배포 기준과 live 배포 확인 결과로 재확인",
    ),
]

print("[최종 확인 결과]")
for check, observed, owner, next_action in final_check_result_packet:
    print(f"- {check}: {observed} | owner={owner} | next={next_action}")


[최종 확인 결과]
- input_shift: none | owner=Data Engineering | next=데이터 수집 경로와 upstream feed 변경 여부 확인
- prediction_shift: high_risk_rate_delta=0.2416 | owner=ML Engineering | next=threshold와 score distribution 재확인
- api_validation: error_rate_delta=0.0667 | owner=Client Integration | next=failed_field와 client payload 변경 확인
- deployment_check: failed_checks=['recall', 'error_rate'], unresolved_risks=['live_deployment'] | owner=QA Lead | next=같은 배포 기준과 live 배포 확인 결과로 재확인


### 5-3. 실패 시 확인 포인트

데이터 파일을 읽지 못하면 먼저 `data/serving_requests_valid.csv`, `data/serving_requests_current.csv`, `data/release_regression_cases.csv`가 Lite files 또는 로컬 repo에 복사되었는지 확인합니다. fallback sample로 실행된 경우 보고서에 sample 기준이라고 적어야 합니다.

최종 report artifact를 읽지 못하면 `artifacts/reports/drift_report.md`, `quality_issue_trace.md`, `release_approval.md`, `ai_qa_checklist.md`가 준비되었는지 확인합니다. 로컬에서는 `uv run --group lab python labs/ch05_qa_strategy/04_build_qa_artifacts.py`로 재생성합니다.

`deployment_ready=True`가 나오더라도 live 배포 확인 결과가 미검증이면 운영 배포 준비 완료라고 쓰지 않습니다. `/health`, `/predict`, Pod readiness, 응답 `model_version`, `threshold` 증거를 별도로 확인해야 합니다.

최종 문장은 원인 확정 문장으로 쓰지 않습니다. 입력 변화, validation failure, latency, prediction shift를 후보로 분리하고 owner별 재평가 조건을 남깁니다.
